In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point
import numpy as np

# 1. Load wildfire polygons and convert from WKT
wildfire_df = pd.read_csv("C:/Users/longl/Downloads/BurnData (2).csv")
wildfire_df['geometry'] = wildfire_df['geometry'].apply(wkt.loads)
wildfire_gdf = gpd.GeoDataFrame(wildfire_df, geometry='geometry', crs="EPSG:4326")

# 2. Load airport locations
airport_df = pd.read_excel("airports_runways_joined.xlsx")
airport_df = airport_df.dropna(subset=['latitude_deg', 'longitude_deg'])

# 3. Get wildfire centroids
wildfire_gdf['centroid'] = wildfire_gdf.geometry.centroid
wildfire_gdf['centroid_lat'] = wildfire_gdf['centroid'].y
wildfire_gdf['centroid_lon'] = wildfire_gdf['centroid'].x

# 4. Haversine distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a)) * 1000  # meters

# 5. Match each wildfire to nearest airport
results = []

for _, fire in wildfire_gdf.iterrows():
    lat1, lon1 = fire['centroid_lat'], fire['centroid_lon']
    distances = haversine(lat1, lon1, airport_df['latitude_deg'], airport_df['longitude_deg'])
    min_idx = np.argmin(distances)
    nearest_airport = airport_df.iloc[min_idx]

    results.append({
        # Wildfire details
        'Fire ID': fire['unique_id'],
        'FIRE LAT': lat1,
        'FIRE LON': lon1,
        'startdate': fire['startdate'],
        'duration': fire['duration'],
        'size (km2)': fire['size (km2)'], 
        'fire_speed (km/day)': fire['fire_speed (km/day)'],  # same conversion

        # Distance
        'DISTANCE': distances[min_idx],  # in meters

        # All airport details
        'ident': nearest_airport['ident'],
        'iata_code': nearest_airport['iata_code'],
        'icao_code': nearest_airport['icao_code'],
        'local_code': nearest_airport['local_code'],
        'CLOSEST_AIRPORT_NAME': nearest_airport['name'],
        'type': nearest_airport['type'],
        'latitude_deg': nearest_airport['latitude_deg'],
        'longitude_deg': nearest_airport['longitude_deg'],
        'elevation_ft': nearest_airport['elevation_ft'],
        'country_name': nearest_airport['country_name'],
        'region_name': nearest_airport['region_name'],
        'runway_lengths_ft': nearest_airport['runway_lengths_ft'],
        'runway_surfaces': nearest_airport['runway_surfaces'],
        'airtanker_base': nearest_airport['airtanker_base']
    })

# 6. Create DataFrame
result_df = pd.DataFrame(results)

# 7. Preview
print(result_df.head())
result_df.to_csv("C:/Users/longl/nearest_airtanker_bases_to_fires_final.csv", index=False)

C:\Users\longl\AppData\Local\Temp\ipykernel_38352\1830277127.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  wildfire_gdf['centroid'] = wildfire_gdf.geometry.centroid


      Fire ID   FIRE LAT   FIRE LON            startdate  duration  \
0  2020_19418  39.949755 -93.231101  2020-02-16 00:00:00        15   
1  2020_19429  39.749156 -97.388993  2020-02-01 00:00:00         4   
2  2020_19427  39.750497 -96.711432  2020-04-03 00:00:00        17   
3  2020_19431  39.733650 -97.399680  2020-02-01 00:00:00         4   
4  2020_19434  39.715525 -97.394927  2020-02-01 00:00:00         4   

   size (km2)  fire_speed (km/day)       DISTANCE ident iata_code  ...  \
0        7.29                 1.61  115059.314201  KGPH       NaN  ...   
1        7.93                 1.67  108924.489030  KSLN       SLN  ...   
2        4.50                 2.56  120320.938839  KTOP       TOP  ...   
3       14.15                 2.00  107048.558942  KSLN       SLN  ...   
4        6.65                 2.44  105161.951419  KSLN       SLN  ...   

                  CLOSEST_AIRPORT_NAME            type latitude_deg  \
0  Midwest National Air Center Airport   small_airport    39.33